# Pregunta 1 — Mediación moderada

**Métodos Cuantitativos de Investigación en Negocios — Tarea 2 (2026)**

## Enunciado de la pregunta 1

Para cada una de las preguntas, describa claramente los pasos seguidos y adjunte los resultados de los análisis efectuados durante el proceso.

**1.** Como parte de una investigación en la que está participando, se le ha solicitado realizar un análisis de un modelo que explica la relación entre la presión competitiva del mercado y el desempeño innovador de la empresa. En este modelo (ver figura 1) se plantea que el efecto de la presión competitiva sobre el desempeño innovador está mediado por la conducta innovadora de los empleados, pero que esta relación depende de la cultura de tolerancia al fracaso de la organización.

![Figura 1. Modelo de mediación moderada](../assets/figura_1_modelo.png)

Las variables del modelo se explican a continuación:

- **Presión Competitiva del Mercado:** Nivel de rivalidad y cambios tecnológicos en el sector, medida en una escala de 1 a 10, donde un mayor número representa una mayor presión competitiva.
- **Cultura de Tolerancia al Fracaso:** Grado en que la gerencia castiga o premia los errores al experimentar, medida en una escala de 1 a 7, donde un mayor número representa una mayor tolerancia al fracaso por parte de la gerencia.
- **Conducta Innovadora del Empleado:** Generación e implementación de ideas nuevas en el trabajo, medida como el número de ideas propuestas por el empleado en los últimos 5 años.
- **Desempeño Innovador de la Empresa:** Generación exitosa de nuevos productos o servicios para el mercado, medida como el porcentaje de ingresos de la organización proveniente de productos o servicios lanzados los últimos 5 años.

Con los datos de las respuestas 200 empleados de 200 empresas distintas, contenidos en la base `Med_Mod.sav`, usted deberá:

**a)** Plantear el modelo en forma de hipótesis.  
**b)** Evaluar cada una de estas hipótesis.  
**c)** Interpretar los resultados.

## Respuesta

El modelo corresponde a una **mediación moderada de primera etapa**: la presión competitiva del mercado puede influir en el desempeño innovador a través de la conducta innovadora del empleado, y ese primer tramo depende de la cultura de tolerancia al fracaso.

En términos de variables estadísticas:

- **X:** Presión competitiva del mercado.
- **W:** Cultura de tolerancia al fracaso.
- **M:** Conducta innovadora del empleado.
- **Y:** Desempeño innovador de la empresa.

La lógica estadística es equivalente al **modelo 7 de Hayes**. En este notebook no se usa la macro PROCESS directamente; se reproduce esa especificación en Python mediante regresiones OLS y bootstrap para el efecto indirecto condicional.


In [1]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pyreadstat
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 160)

# Carpeta de datos.
# El repositorio deja las bases SPSS en Tarea2/data para que todos los notebooks sean reproducibles.
# La lógica revisa varias ubicaciones habituales: ejecución desde Tarea2/notebooks, desde Tarea2, desde la raíz del repo o desde Colab.
candidatas = [
    Path.cwd() / 'data',
    Path.cwd().parent / 'data',
    Path.cwd() / 'Tarea2' / 'data',
    Path.cwd()  # respaldo: bases junto al notebook (por ejemplo, en Colab tras subirlas)
]
DATA_DIR = next((ruta for ruta in candidatas if (ruta / 'Med_Mod.sav').exists()), candidatas[0])

print('Directorio de datos:', DATA_DIR)


Directorio de datos: /home/claude/final/data


## 1.a) Planteamiento del modelo en forma de hipótesis

Se analiza una **mediación moderada de primera etapa**:

- $X$: presión competitiva del mercado.
- $M$: conducta innovadora del empleado.
- $Y$: desempeño innovador de la empresa.
- $W$: cultura de tolerancia al fracaso.

La moderación se ubica en la relación $X\rightarrow M$. El análisis se articula en **tres ecuaciones**:

$$\text{(1) Mediador:}\quad M=i_M+a_1X+a_2W+a_3XW+e_M$$
$$\text{(2) Resultado:}\quad Y=i_Y+c'X+bM+e_Y$$
$$\text{(3) Efecto total:}\quad Y=i_T+cX+e_T$$

Las ecuaciones (1) y (2) constituyen el sistema estructural equivalente al modelo 7 de Hayes; la ecuación (3) es la que permite evaluar H1 (efecto total). A partir de (1) y (2) se deriva el efecto indirecto condicional $(a_1+a_3W)b$, que es una función de los coeficientes y no una regresión adicional. Las hipótesis son: H1, la presión competitiva tiene un efecto total positivo sobre desempeño; H2, presión competitiva aumenta conducta innovadora; H3, conducta innovadora aumenta desempeño controlando presión; H4, existe mediación; H5, tolerancia al fracaso fortalece $X\rightarrow M$; y H6, el efecto indirecto aumenta con la tolerancia. H4 pregunta si existe un mecanismo indirecto; H6 pregunta si ese mecanismo cambia de intensidad según $W$. El camino directo $c'$ también se estima porque aparece en la figura.

In [2]:
# Carga de la base SPSS y revisión inicial de variables.
# apply_value_formats=False conserva los valores numéricos originales, necesarios para estimar los modelos.
df, meta = pyreadstat.read_sav(DATA_DIR/'Med_Mod.sav', apply_value_formats=False)

print('Dimensión:', df.shape)
print(df.describe().round(3).to_string())

print('\nEtiquetas de variables:')
for c in df.columns:
    print(c, '->', meta.column_names_to_labels.get(c))


Dimensión: (200, 5)
            ID  Pres_Comp  Cult_Tol_Frac  Con_Inn_Emple  Des_Inn_Empre
count  200.000    200.000        200.000        200.000        200.000
mean   100.500      4.576          2.785          9.595          9.639
std     57.879      1.079          0.758          3.406          2.983
min      1.000      2.129          1.054          3.000          4.113
25%     50.750      3.853          2.200          7.000          7.714
50%    100.500      4.541          2.892          9.000          9.304
75%    150.250      5.328          3.451         12.000         11.445
max    200.000      7.778          4.543         21.000         21.066

Etiquetas de variables:
ID -> None
Pres_Comp -> Presión Competitiva del Mercado
Cult_Tol_Frac -> Cultura de Tolerancia al Fracaso
Con_Inn_Emple -> Conducta Innovadora del Empleado
Des_Inn_Empre -> Desempeño Innovador de la Empresa


## 1.b) Evaluación de las hipótesis: preparación de variables

Se centran $X$ y $W$ en sus medias. Esto no cambia el ajuste ni el término de interacción, pero hace que los efectos principales se interpreten cuando la otra variable está en su media y reduce colinealidad no esencial.

In [3]:
# Definición de variables del modelo.
# X: presión competitiva; W: tolerancia al fracaso; M: conducta innovadora; Y: desempeño innovador.
X, W, M, Y = 'Pres_Comp', 'Cult_Tol_Frac', 'Con_Inn_Emple', 'Des_Inn_Empre'

# Se centran X y W para interpretar los efectos principales en el nivel promedio del moderador.
# La interacción XW representa el término de moderación del primer tramo X -> M.
df['Xc'] = df[X] - df[X].mean()
df['Wc'] = df[W] - df[W].mean()
df['XW'] = df['Xc'] * df['Wc']

print(df[[X, W, 'Xc', 'Wc', 'XW']].head().round(3).to_string(index=False))


 Pres_Comp  Cult_Tol_Frac    Xc     Wc     XW
     5.745          3.358 1.169  0.573  0.669
     4.793          3.561 0.217  0.776  0.168
     5.972          4.083 1.396  1.298  1.811
     7.285          4.054 2.709  1.269  3.436
     4.649          1.622 0.073 -1.163 -0.085


## 1.b) Evaluación de las hipótesis: estimación de las tres ecuaciones

Se estiman las tres ecuaciones del planteamiento: (1) la del mediador, donde el coeficiente de `XW` evalúa si la pendiente de presión competitiva cambia según la tolerancia al fracaso; (2) la del resultado, que incorpora simultáneamente presión competitiva y conducta innovadora; y (3) la del efecto total, necesaria para H1. Las dos primeras forman el sistema del modelo 7; la tercera no modifica el modelo, sino que reporta el efecto total requerido para interpretar la mediación.

Se utilizan errores estándar robustos HC3. En cada tabla se interpretan el coeficiente, su error estándar, el estadístico de contraste y el p-value.

In [4]:
# Especificación equivalente al modelo 7 de Hayes, estimada en Python con regresiones OLS.
# 1) Modelo del mediador: evalúa X -> M y la interacción XW.
# 2) Modelo del resultado: evalúa M -> Y controlando X.
# 3) Modelo total: evalúa el efecto total X -> Y antes de introducir el mediador.
modelo_M = smf.ols(f'{M} ~ Xc + Wc + XW', df).fit(cov_type='HC3')
modelo_Y = smf.ols(f'{Y} ~ Xc + {M}', df).fit(cov_type='HC3')
modelo_total = smf.ols(f'{Y} ~ Xc', df).fit(cov_type='HC3')

print('MODELO DEL MEDIADOR\n', modelo_M.summary().as_text())
print('\nMODELO DEL RESULTADO\n', modelo_Y.summary().as_text())
print('\nEFECTO TOTAL\n', modelo_total.summary().as_text())

# Tabla resumida de los coeficientes clave para leer directamente las hipótesis H2 y H5.
coeficientes_M = pd.DataFrame({
    'Coeficiente': modelo_M.params,
    'Error estándar HC3': modelo_M.bse,
    'Estadístico z': modelo_M.tvalues,
    'p-value': modelo_M.pvalues
})
print('\nCOEFICIENTES CENTRALES DEL MODELO DEL MEDIADOR')
print(coeficientes_M.loc[['Xc', 'Wc', 'XW']].round(4).to_string())

# El 0.5248 sale del coeficiente de XW: es cuánto cambia la pendiente de X -> M
# cuando la tolerancia al fracaso aumenta una unidad.
print('\nEcuación estimada del mediador:')
p_ = modelo_M.params
print(f'M_hat = {p_["Intercept"]:.4f} + {p_["Xc"]:.4f}·Xc + {p_["Wc"]:.4f}·Wc + {p_["XW"]:.4f}·(Xc×Wc)')


MODELO DEL MEDIADOR
                             OLS Regression Results                            
Dep. Variable:          Con_Inn_Emple   R-squared:                       0.981
Model:                            OLS   Adj. R-squared:                  0.980
Method:                 Least Squares   F-statistic:                     2625.
Date:                Thu, 09 Jul 2026   Prob (F-statistic):          6.32e-158
Time:                        04:17:22   Log-Likelihood:                -134.43
No. Observations:                 200   AIC:                             276.9
Df Residuals:                     196   BIC:                             290.0
Df Model:                           3                                         
Covariance Type:                  HC3                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      9.5207      0.03

### Estadístico de comparación y sustitución de los contrastes

Con errores estándar robustos HC3 la inferencia es asintótica, por lo que el estadístico de comparación bilateral al 95% de confianza es la normal estándar, $z_c=\pm 1.9600$. La regla de decisión es: se rechaza $H_0$ si $|z|>1.9600$, equivalente a $p<0.05$.

A modo de ilustración, para la interacción (H5) el estadístico se calcula como el coeficiente dividido por su error estándar robusto:

$$z=\frac{a_3}{EE(a_3)}=\frac{0.5248}{0.0389}=13.48.$$

Como $13.48>1.9600$ y $p<0.001$, se rechaza $H_0: a_3=0$: existe moderación. La misma regla aplica a cada coeficiente de las tres ecuaciones. La celda siguiente calcula el valor crítico y las pendientes condicionales que luego alimentan los efectos indirectos: cada una se obtiene reemplazando en $a(W)=a_1+a_3W$ con $DE(W)=0.7583$.

In [5]:
# Estadístico de comparación (95%, bilateral) y pendientes condicionales de X sobre M.
# El valor crítico proviene de la normal estándar porque HC3 hace la inferencia asintótica.
zc = stats.norm.ppf(.975)
z_a3 = modelo_M.params['XW'] / modelo_M.bse['XW']
print(f'z crítico bilateral (95%) = ±{zc:.4f}')
print(f'z interacción = {modelo_M.params["XW"]:.4f}/{modelo_M.bse["XW"]:.4f} = {z_a3:.4f} -> se rechaza H0: a3=0')

sdW = df['Wc'].std(ddof=1)
print(f'\nDE(W) = {sdW:.4f}')
print(f'a(baja)  = a1 + a3*(-DE) = {modelo_M.params["Xc"] - modelo_M.params["XW"]*sdW:.4f}')
print(f'a(media) = a1            = {modelo_M.params["Xc"]:.4f}')
print(f'a(alta)  = a1 + a3*(+DE) = {modelo_M.params["Xc"] + modelo_M.params["XW"]*sdW:.4f}')

z crítico bilateral (95%) = ±1.9600
z interacción = 0.5248/0.0389 = 13.4753 -> se rechaza H0: a3=0

DE(W) = 0.7583
a(baja)  = a1 + a3*(-DE) = 1.6029
a(media) = a1            = 2.0008
a(alta)  = a1 + a3*(+DE) = 2.3988


### Relación entre las tres ecuaciones: por qué $c \neq c' + a_1 b$ en el modelo 7

En el modelo 4 (mediación simple) se cumple exactamente la descomposición $c=c'+ab$. En el modelo 7 esa identidad **no se cumple con $a_1$**, porque $a_1$ es el efecto de $X$ sobre $M$ condicional a $W$ y a la interacción, mientras que la ecuación del efecto total omite a $W$. La identidad OLS exacta que sí se cumple es $c=c'+b\,\tilde a_1$, donde $\tilde a_1$ proviene de la regresión simple $M\sim X$. La brecha entre $\tilde a_1$ y $a_1$ se explica por la correlación entre $X$ y $W$. Además, el efecto total implicado por el sistema es condicional: $c(W)=c'+(a_1+a_3W)b$. Por eso la evidencia formal de mediación no descansa en $c$, sino en los efectos indirectos condicionales y su bootstrap.

In [6]:
# Verificación numérica de la descomposición del efecto total.
modelo_M_simple = smf.ols(f'{M} ~ Xc', df).fit()   # M ~ X sin W ni XW
a1s = modelo_M_simple.params['Xc']
c_  = modelo_total.params['Xc']
cp_ = modelo_Y.params['Xc']
b_  = modelo_Y.params[M]
a1_ = modelo_M.params['Xc']; a3_ = modelo_M.params['XW']

print(f'c (efecto total)                   = {c_:.4f}')
print(f"c' + a1*b (descomposición modelo 4) = {cp_ + a1_*b_:.4f}  -> NO coincide")
print(f'a1_simple (M ~ X)                  = {a1s:.4f}')
print(f"c' + b*a1_simple (identidad OLS)    = {cp_ + b_*a1s:.4f}  -> coincide exactamente")
print(f'corr(Xc,Wc) = {df.Xc.corr(df.Wc):.4f}  (explica la brecha entre a1 y a1_simple)')
print(f"Efecto total condicional c(W) = c' + (a1+a3*W)*b en W medio: {cp_ + a1_*b_:.4f}")

c (efecto total)                   = 2.0281
c' + a1*b (descomposición modelo 4) = 1.7427  -> NO coincide
a1_simple (M ~ X)                  = 2.3336
c' + b*a1_simple (identidad OLS)    = 2.0281  -> coincide exactamente
corr(Xc,Wc) = 0.1739  (explica la brecha entre a1 y a1_simple)
Efecto total condicional c(W) = c' + (a1+a3*W)*b en W medio: 1.7427


## 1.b) Evaluación de las hipótesis: efectos indirectos condicionales y bootstrap

Con los coeficientes estimados se calcula el efecto indirecto condicional: $(a_1+a_3W)b$. Este efecto se evalúa en tres niveles de tolerancia al fracaso: baja ($-1DE$), media y alta ($+1DE$).

El bootstrap de 5.000 remuestras se usa porque el producto de coeficientes no necesariamente sigue una distribución normal; esta cantidad sigue una práctica habitual en aplicaciones tipo PROCESS. La regla de decisión es: si el intervalo de confianza al 95% no contiene cero, el efecto indirecto se interpreta como significativo. El índice de mediación moderada se calcula como $a_3b$ y resume cuánto cambia el efecto indirecto cuando aumenta la tolerancia al fracaso.


In [7]:
# Coeficientes necesarios para el efecto indirecto condicional.
# a1: efecto de X sobre M cuando W está en su media.
# a3: interacción XW; indica cómo cambia a1 según W.
# b: efecto de M sobre Y controlando X.
a1 = modelo_M.params['Xc']
a3 = modelo_M.params['XW']
b = modelo_Y.params[M]

sw = df['Wc'].std(ddof=1)
niveles = {'Baja (-1 DE)': -sw, 'Media': 0, 'Alta (+1 DE)': sw}

# Bootstrap: se remuestrean filas completas para conservar la estructura conjunta de X, W, M e Y.
rng = np.random.default_rng(20260706)
boot = []
for _ in range(5000):
    d = df.iloc[rng.integers(0, len(df), len(df))]
    mm = smf.ols(f'{M} ~ Xc + Wc + XW', d).fit()
    my = smf.ols(f'{Y} ~ Xc + {M}', d).fit()
    boot.append((mm.params['Xc'], mm.params['XW'], my.params[M]))
boot = np.asarray(boot)

filas = []
for nombre, w in niveles.items():
    vals = (boot[:, 0] + boot[:, 1] * w) * boot[:, 2]
    filas.append([nombre, a1 + a3 * w, (a1 + a3 * w) * b, *np.quantile(vals, [.025, .975])])

tabla = pd.DataFrame(filas, columns=['Tolerancia', 'Efecto X→M', 'Indirecto', 'IC95% inferior', 'IC95% superior'])
indice = a3 * b
ici = np.quantile(boot[:, 1] * boot[:, 2], [.025, .975])

print(tabla.round(4).to_string(index=False))
print(f'\nÍndice de mediación moderada = {indice:.4f}; IC95% bootstrap [{ici[0]:.4f}, {ici[1]:.4f}]')


  Tolerancia  Efecto X→M  Indirecto  IC95% inferior  IC95% superior
Baja (-1 DE)      1.6029     1.3748          1.2880          1.4620
       Media      2.0008     1.7162          1.6227          1.8104
Alta (+1 DE)      2.3988     2.0576          1.9319          2.1853

Índice de mediación moderada = 0.4502; IC95% bootstrap [0.3758, 0.5184]


## 1.b) Evaluación de las hipótesis: diagnósticos del modelo

Se revisa multicolinealidad mediante VIF y heterocedasticidad con Breusch--Pagan. Los VIF permiten verificar que el centrado redujo colinealidad no esencial entre $X$, $W$ y la interacción. Además, las regresiones principales usan errores estándar robustos HC3, por lo que las conclusiones son más seguras frente a heterocedasticidad.


In [8]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

# VIF bajo indica que la interacción centrada no genera multicolinealidad problemática.
vif = pd.DataFrame({
    'Variable': modelo_M.model.exog_names,
    'VIF': [variance_inflation_factor(modelo_M.model.exog, i) for i in range(modelo_M.model.exog.shape[1])]
})

# Breusch--Pagan revisa heterocedasticidad; la inferencia principal ya usa HC3.
bp = het_breuschpagan(modelo_M.resid, modelo_M.model.exog)
print(vif.round(3).to_string(index=False))
print(f'\nBreusch–Pagan: LM={bp[0]:.4f}, p={bp[1]:.4g}')


 Variable   VIF
Intercept 1.027
       Xc 1.036
       Wc 1.046
       XW 1.016

Breusch–Pagan: LM=1.1407, p=0.7673


## 1.c) Interpretación de los resultados

La presión competitiva aumenta significativamente la conducta innovadora ($a_1=2.0008$), y la conducta innovadora predice el desempeño ($b=0.8577$). El efecto directo de presión competitiva deja de ser significativo al incorporar el mediador ($c'=0.0265$, $p=0.590$), compatible con mediación completa en la media de $W$.

La interacción es positiva y significativa ($a_3=0.5248$, $p<0.001$): una cultura tolerante al fracaso fortalece el efecto de la presión competitiva sobre la conducta innovadora. Los efectos indirectos son 1.3748, 1.7162 y 2.0576 para tolerancia baja, media y alta; todos sus IC95% excluyen cero. El índice de mediación moderada es 0.4502, IC95% [0.3758, 0.5184]. **Se apoyan las hipótesis de mediación y moderación: el efecto indirecto crece cuando la tolerancia al fracaso es mayor.**